# geo0641 – From Open Data to a Map: DWD Climate Stations Pipeline

**Unit:** geo0640 DWD CDC Stations & Time Series

In this notebook we build a complete, small geospatial data pipeline:

```
DWD Open Data (HTTPS)  →  raw text  →  pandas DataFrame  →  GeoDataFrame  →  GeoPackage  →  QGIS / web map
        acquire             inspect        parse & clean        georeference      persist        view & check
```

**Learning goals.** After this notebook you can
1. download an open dataset reproducibly via HTTPS and cache it locally,
2. inspect an unknown text format and parse it robustly (fixed-width, encoding, dates),
3. filter tabular data with boolean masks,
4. turn coordinates into geometries with a defined CRS,
5. persist the result as a GeoPackage and inspect it in QGIS on top of an OGC WMS basemap.

> Motto of the course: **Python does the work, PostGIS keeps the data, QGIS lets you see, check and interact.**

## 0. Setup
We use only well-known packages. On the EOLab JupyterHub everything is pre-installed.

In [1]:
from pathlib import Path

import requests
import pandas as pd
import geopandas as gpd

print("pandas", pd.__version__, "| geopandas", gpd.__version__)

pandas 3.0.6 | geopandas 1.2.0


In [2]:
# Where do the data live?
# On the JupyterHub a read-only shared copy may exist; otherwise we download into ./data.
SHARED_DIR = Path("/srv/data/dwd/cdc/annual/kl/historical")
LOCAL_DIR = Path("data/original/dwd/annual/kl/historical")
OUT_DIR = Path("data/derived")
LOCAL_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = ("https://opendata.dwd.de/climate_environment/CDC/"
            "observations_germany/climate/annual/kl/historical/")
STATION_FILE = "KL_Jahreswerte_Beschreibung_Stationen.txt"

## 1. Acquire: download once, then work offline
Open the [DWD CDC directory](https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/annual/kl/historical/) in your browser.
It is a plain HTML directory listing. The station description file lists all stations of the annual climate (KL) collection.

**Good practice:** download into `data/original/` and never edit original files. Everything we produce goes to `data/derived/`.

In [3]:
def fetch(url: str, dest: Path, overwrite: bool = False) -> Path:
    """Download url to dest unless a local copy exists. Returns the local path."""
    if dest.exists() and not overwrite:
        print(f"cached: {dest}")
        return dest
    r = requests.get(url, timeout=60)
    r.raise_for_status()
    dest.write_bytes(r.content)
    print(f"downloaded {len(r.content):,} bytes -> {dest}")
    return dest


shared = SHARED_DIR / STATION_FILE
station_path = shared if shared.exists() else fetch(BASE_URL + STATION_FILE, LOCAL_DIR / STATION_FILE)
station_path

cached: data/original/dwd/annual/kl/historical/KL_Jahreswerte_Beschreibung_Stationen.txt


PosixPath('data/original/dwd/annual/kl/historical/KL_Jahreswerte_Beschreibung_Stationen.txt')

## 2. Inspect: look at the raw bytes before you parse
**Predict:** what kind of file is this? Which separator? Which encoding? Which problems do you expect?

In [4]:
raw = station_path.read_bytes()
print(raw[:400])

b'Stations_id von_datum bis_datum Stationshoehe geoBreite geoLaenge Stationsname Bundesland Abgabe\r\n----------- --------- --------- ------------- --------- --------- ----------------------------------------- ---------- ------\r\n00001 19310101 19851231            478     47.8413    8.8493 Aach                                     Baden-W\xfcrttemberg                        Frei                            '


In [5]:
for line in raw.decode("latin-1").splitlines()[:6]:
    print(repr(line[:160]))

'Stations_id von_datum bis_datum Stationshoehe geoBreite geoLaenge Stationsname Bundesland Abgabe'
'----------- --------- --------- ------------- --------- --------- ----------------------------------------- ---------- ------'
'00001 19310101 19851231            478     47.8413    8.8493 Aach                                     Baden-Württemberg                        Frei             '
'00003 18510101 20101231            202     50.7827    6.0941 Aachen                                   Nordrhein-Westfalen                      Frei             '
'00044 19720101 20251231             44     52.9336    8.2370 Großenkneten                             Niedersachsen                            Frei             '
'00052 19730101 20011231             46     53.6623   10.1990 Ahrensburg-Wulfsdorf                     Schleswig-Holstein                       Frei             '


Observations to discuss:
* **Fixed-width** columns, no separator. Station names contain blanks (`Aachen-Orsbach`, `Bergzabern, Bad`), so `split()` is not safe.
* **Encoding** is ISO-8859-1 (Latin-1): `Württemberg`.
* **Dates** as `YYYYMMDD` strings.
* Line 2 is a row of dashes that *looks* like a column-width specification …

## 3. Parse: a robust reader for the DWD station list
### 3.1 Does the dashed header tell the truth?
Let us measure where the values really start and end in a data line and compare with the dashes.

In [6]:
import re

lines = raw.decode("latin-1").splitlines()
print("header dashes:")
for m in re.finditer(r"-+", lines[1]):
    print(f"  {m.start():3d}-{m.end():3d}")
print("tokens of a data line:")
for m in re.finditer(r"\S+", lines[3]):
    print(f"  {m.start():3d}-{m.end():3d}  {m.group()}")

header dashes:
    0- 11
   12- 21
   22- 31
   32- 45
   46- 55
   56- 65
   66-107
  108-118
  119-125
tokens of a data line:
    0-  5  00003
    6- 14  18510101
   15- 23  20101231
   35- 38  202
   43- 50  50.7827
   54- 60  6.0941
   61- 67  Aachen
  102-121  Nordrhein-Westfalen
  143-147  Frei


The dashes do **not** match the data: e.g. the station id has 11 dashes but 5 digits, and the state column is much wider than its 10 dashes.
Lesson: **check the data, not the documentation.** We define the column positions ourselves.

### 3.2 Exercise: read the file with `pd.read_fwf`
Use the column positions below, the English column names, skip the two header lines, and choose the right encoding.
Keep the dates as strings first (`dtype`), we convert them afterwards.
Read the **station id as text** too: it is an identifier, not a number. As text, `00044` stays `00044` (and the Zugspitze stays `05792`),
so the id matches the DWD file names (`…_00044_…`) and other tables that use the same key. Rule of this course: identifiers are text.

In [7]:
NAMES = ["station_id", "date_from", "date_to", "altitude", "lat", "lon", "name", "state", "access"]
COLSPECS = [(0, 5), (6, 14), (15, 23), (24, 38), (39, 50), (51, 60), (61, 102), (102, 143), (143, None)]

df = pd.read_fwf(
    station_path,
    colspecs=COLSPECS,
    names=NAMES,
    skiprows=2,
    encoding="latin-1",
    dtype={"station_id": str, "date_from": str, "date_to": str},
)
df.head()

,station_id,date_from,date_to,altitude,lat,lon,name,state,access
0,00001,19310101,19851231,478,47.8413,8.8493,Aach,Baden-Württemberg,Frei
1,00003,18510101,20101231,202,50.7827,6.0941,Aachen,Nordrhein-Westfalen,Frei
2,00044,19720101,20251231,44,52.9336,8.2370,Großenkneten,Niedersachsen,Frei
3,00052,19730101,20011231,46,53.6623,10.1990,Ahrensburg-Wulfsdorf,Schleswig-Holstein,Frei
4,00061,19760101,19771231,339,48.8443,12.6171,Aiterhofen,Bayern,Frei


In [8]:
# Convert types and use the station id as index
df["date_from"] = pd.to_datetime(df["date_from"], format="%Y%m%d")
df["date_to"] = pd.to_datetime(df["date_to"], format="%Y%m%d")
df = df.set_index("station_id")
df.dtypes

date_from    datetime64[us]
date_to      datetime64[us]
altitude              int64
lat                 float64
lon                 float64
name                    str
state                   str
access                  str
dtype: object

### 3.3 Quality checks
Never skip this step. How many stations? Missing values? Plausible value ranges?

In [9]:
print(len(df), "stations")
print(df.isna().sum())
df[["altitude", "lat", "lon"]].describe()

1226 stations
date_from     0
date_to       0
altitude      0
lat           0
lon           0
name          0
state         0
access       22
dtype: int64


,altitude,lat,lon
count,1226.000000,1226.000000,1226.000000
mean,306.826264,50.663976,10.148277
std,281.525674,1.923107,2.012521
min,0.000000,47.398400,6.024400
25%,80.000000,49.083000,8.576175
50%,256.500000,50.642400,10.008600
75%,457.000000,51.964125,11.686500
max,2956.000000,55.011000,14.950600


In [10]:
df["state"].value_counts()

state
Bayern                    246
Baden-Württemberg         200
Niedersachsen             119
Nordrhein-Westfalen        94
Hessen                     91
Thüringen                  84
Schleswig-Holstein         69
Sachsen                    61
Rheinland-Pfalz            60
Sachsen-Anhalt             57
Brandenburg                46
Mecklenburg-Vorpommern     42
Berlin                     21
Saarland                   19
Hamburg                    12
Bremen                      5
Name: count, dtype: int64

**Discuss:** some `access` values are missing (NaN). Would a whitespace-split parser have noticed that, or silently shifted the columns?

## 4. Filter: stations in NRW, still active, with long records
Criteria:
1. state is *Nordrhein-Westfalen*,
2. still active: `date_to` equals the most recent end date in the file,
3. record started in 1970 or earlier.

In [11]:
latest = df["date_to"].max()
mask = (
    (df["state"] == "Nordrhein-Westfalen")
    & (df["date_to"] == latest)
    & (df["date_from"].dt.year <= 1970)
)
df_nrw = df[mask].copy()
print(f"{len(df_nrw)} stations, latest end date: {latest:%Y-%m-%d}")
df_nrw[["name", "altitude", "date_from", "date_to"]]

16 stations, latest end date: 2025-12-31


,name,altitude,date_from,date_to
station_id,,,,
00555,Bochum,110,1925-01-01,2025-12-31
01078,Düsseldorf,37,1940-01-01,2025-12-31
01300,Eslohe,351,1931-01-01,2025-12-31
01303,Essen-Bredeney,150,1888-01-01,2025-12-31
01327,Weilerswist-Lommersum,147,1937-01-01,2025-12-31
01590,Geldern-Walbeck,37,1937-01-01,2025-12-31
02483,Kahler Asten,839,1927-01-01,2025-12-31
02497,Kall-Sistig,505,1937-01-01,2025-12-31
02629,Kleve,46,1851-01-01,2025-12-31


## 5. Georeference: from lat/lon columns to geometries
**Predict:** in which coordinate reference system (CRS) are `lat`/`lon` given? What goes wrong if we forget to declare it?

In [12]:
gdf_nrw = gpd.GeoDataFrame(
    df_nrw,
    geometry=gpd.points_from_xy(df_nrw["lon"], df_nrw["lat"]),
    crs="EPSG:4326",
)
gdf_nrw.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

For NRW we work in **ETRS89 / UTM zone 32N (EPSG:25832)**: metric coordinates, used by all official NRW geodata.
CRS are treated in depth in unit geo0220. For now: re-project and compare the coordinate values.

In [13]:
gdf_nrw_25832 = gdf_nrw.to_crs("EPSG:25832")
gdf_nrw_25832[["name", "geometry"]].head()

,name,geometry
station_id,,
00555,Bochum,POINT (377074.479 5707205.469)
01078,Düsseldorf,POINT (344430.456 5685106.643)
01300,Eslohe,POINT (441136.149 5678409.213)
01303,Essen-Bredeney,POINT (358643.171 5696723.636)
01327,Weilerswist-Lommersum,POINT (344009.735 5620116.295)


## 6. Persist and view
### 6.1 GeoPackage
A GeoPackage is a single SQLite file holding one or more layers: the standard exchange format of this course.

In [14]:
gpkg = OUT_DIR / "dwd_stations.gpkg"
gdf_all = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df["lon"], df["lat"]), crs="EPSG:4326").to_crs("EPSG:25832")
gdf_all.to_file(gpkg, layer="kl_stations_all", driver="GPKG")
gdf_nrw_25832.to_file(gpkg, layer="kl_stations_nrw_active", driver="GPKG")
gpd.list_layers(gpkg)

,name,geometry_type
0,rr_stations_nrw_active,Point
1,kl_stations_all,Point
2,kl_stations_nrw_active,Point


### 6.2 QGIS: view and check
1. Download `data/derived/dwd_stations.gpkg` from the JupyterHub (right-click → Download) and drag it into QGIS.
2. Set the project CRS to **EPSG:25832**.
3. Add the NRW topographic map as **WMS** (*Layer → Add Layer → Add WMS/WMTS Layer → New*):
   `https://www.wms.nrw.de/geobasis/wms_nw_dtk` (alternative orthophotos: `https://www.wms.nrw.de/geobasis/wms_nw_dop`).
4. Check: are all active stations inside NRW? Does *Aachen* sit on Aachen? Label the points with `station_id` and `name`.

QGIS here is the **viewer and quality check** of what our code produced, and the **OGC client** for the basemap.

### 6.3 Quick web map in the notebook

In [15]:
gdf_nrw.explore(column="altitude", cmap="viridis", tooltip=["name", "altitude", "date_from"],
                marker_kwds={"radius": 6})

## 7. Exercise (pairs): transfer to another DWD product
The hourly precipitation network (used again in unit geo0650) has its own station list:

`https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/hourly/precipitation/recent/RR_Stundenwerte_Beschreibung_Stationen.txt`

1. Download it with `fetch()` into a suitable folder below `data/original/dwd/`.
2. Inspect it. Are the column positions the same? Check, do not assume.
3. Select all **active** precipitation stations in NRW and add them as a third layer `rr_stations_nrw_active` to the GeoPackage.
4. In QGIS: how does the precipitation network compare with the climate network in density?

In [16]:
RR_URL = ("https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/"
          "hourly/precipitation/recent/RR_Stundenwerte_Beschreibung_Stationen.txt")
rr_dir = Path("data/original/dwd/hourly/precipitation/recent")
rr_dir.mkdir(parents=True, exist_ok=True)
rr_path = fetch(RR_URL, rr_dir / "RR_Stundenwerte_Beschreibung_Stationen.txt")

rr_lines = rr_path.read_bytes().decode("latin-1").splitlines()
for m in re.finditer(r"\S+", rr_lines[3]):      # verify positions before reusing COLSPECS
    print(f"{m.start():3d}-{m.end():3d}  {m.group()}")

rr = pd.read_fwf(rr_path, colspecs=COLSPECS, names=NAMES, skiprows=2, encoding="latin-1",
                 dtype={"station_id": str, "date_from": str, "date_to": str})
rr["date_from"] = pd.to_datetime(rr["date_from"], format="%Y%m%d")
rr["date_to"] = pd.to_datetime(rr["date_to"], format="%Y%m%d")
rr = rr.set_index("station_id")
rr_nrw = rr[(rr["state"] == "Nordrhein-Westfalen") & (rr["date_to"] == rr["date_to"].max())]
gdf_rr = gpd.GeoDataFrame(rr_nrw, geometry=gpd.points_from_xy(rr_nrw["lon"], rr_nrw["lat"]),
                          crs="EPSG:4326").to_crs("EPSG:25832")
gdf_rr.to_file(gpkg, layer="rr_stations_nrw_active", driver="GPKG")
print(len(gdf_rr), "active hourly precipitation stations in NRW")


cached: data/original/dwd/hourly/precipitation/recent/RR_Stundenwerte_Beschreibung_Stationen.txt
  0-  5  00020
  6- 14  20040814
 15- 23  20261007
 35- 38  432
 43- 50  48.9219
 54- 60  9.9129
 61- 85  Abtsgmünd-Untergröningen
102-119  Baden-Württemberg
143-147  Frei


218 active hourly precipitation stations in NRW


## 8. Stretch (optional): from notebook code to a reusable function
Write `read_dwd_station_list(path) -> pd.DataFrame` that encapsulates §3, including the type conversion and a sanity check
(e.g. raise an error if more than 1 % of the coordinates are missing). Write two small `assert`-based tests.
This function will later move into the course package `opengeo.dwd`.

In [17]:
def read_dwd_station_list(path: Path) -> pd.DataFrame:
    """Read a DWD CDC '*_Beschreibung_Stationen.txt' file into a typed DataFrame."""
    d = pd.read_fwf(path, colspecs=COLSPECS, names=NAMES, skiprows=2, encoding="latin-1",
                    dtype={"station_id": str, "date_from": str, "date_to": str})
    for c in ("date_from", "date_to"):
        d[c] = pd.to_datetime(d[c], format="%Y%m%d")
    if d[["lat", "lon"]].isna().mean().max() > 0.01:
        raise ValueError("more than 1 % missing coordinates: check the column positions")
    return d.set_index("station_id")


t = read_dwd_station_list(station_path)
assert t.index.is_unique and (t.index.str.len() == 5).all()   # ids as 5-character text
assert t["lat"].between(47, 56).all() and t["lon"].between(5, 16).all()
print("tests passed")


tests passed


## 9. Wrap-up
**Exit ticket (2 min):** what was the *muddiest point* today? Write one sentence in the course chat or on a card.

**Follow-up:**
1. Finish §7.
2. Make an annotated QGIS map (EPSG:25832, NRW WMS basemap, labels `station_id – name`, legend, scale bar, title, data source *Deutscher Wetterdienst*) of the active NRW climate and precipitation stations.

**Homework and presentations:** see the course plan in the README of your course repository (it differs between courses).
